<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-title.png?v=6966887b41" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-review-roadmap.png?v=fad8e5d501" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-review-rlhf.png?v=2171b7d8a2" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-review-memory.png?v=0e83291c30" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-assistant.png?v=ad72440f9b" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-setup.1.png?v=e0d4ac08dc" class="img-responsive"/>
</div>

In [ ]:
# irishQwen (solutions branch) and its requirements
import os, sys, subprocess

def run(cmd, cwd=None):
    """Run a shell command, stop if it fails."""
    subprocess.run(cmd, cwd=cwd, check=True)

IQ = "/content/irishQwen"
URL = "https://github.com/wtheisen/irishQwen.git"
if not os.path.exists(IQ):
    run(["git", "clone", "--depth", "1",
         "--branch", "solutions", URL, IQ])
run([sys.executable, "-m", "pip", "install", "-q",
     "-r", f"{IQ}/requirements.txt"])
sys.path.insert(0, IQ)

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-convert.1.png?v=8a30e5dfcc" class="img-responsive"/>
</div>

In [ ]:
# Qwen3-0.6B, the chat model, as irishQwen (~3 min)
import torch
print("GPU:", torch.cuda.get_device_name(0))

CKPT = "/content/qwen3_0.6b.pt"
TOK_DIR = "/content/qwen3_tokenizer"
if not os.path.exists(CKPT):
    run([sys.executable, "scripts/import_qwen3.py",
         "--model", "Qwen/Qwen3-0.6B",
         "--out", CKPT, "--tokenizer_dir", TOK_DIR],
        cwd=IQ)

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-load.1.png?v=9d91c8905a" class="img-responsive"/>
</div>

In [ ]:
from irishQwen.irishQwen import IrishQwen
from irishQwen.qwen_tokenizer import QwenTokenizer
from irishQwen import optimizers

tok = QwenTokenizer(f"{TOK_DIR}/tokenizer.json")
model = IrishQwen.from_checkpoint_config(
    CKPT, ctx_len=512)
model.load_converted_qwen3_checkpoint(CKPT)
PAD = model.padding_idx
IM_END = tok.special_tokens["<|im_end|>"]

def ask(question, n=60):
    """Chat template, thinking off, greedy."""
    prompt = tok.chat_prompt(question)
    out = model.chat(prompt, max_new_tokens=n,
                     temperature=0, eos_token_id=IM_END)
    text = tok.decode(out[len(prompt):])
    return text.replace("<|im_end|>", "").strip()

print(ask("When is Exam 02 in CSE 10124?"))

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-rag-why.png?v=0d86bea88c" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-rag.png?v=0296e280c7" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-docs.1.png?v=4a1290b16f" class="img-responsive"/>
</div>

In [ ]:
# Our documents: the CSE 10124 schedule, a row each
import csv, io, urllib.request
from datetime import datetime

SHEET = ("https://docs.google.com/spreadsheets/d/e/"
         "2PACX-1vRKFSFriFOBvLteVPnMotBJ2vDKXaolJiLM2X"
         "mApjQ8JgUkANBK0uzlaOIZH5GZNlG5jBS77JuVo1Ik"
         "/pub?gid=1012833739&single=true"
         "&output=csv")
text = urllib.request.urlopen(SHEET).read().decode()
rows = list(csv.reader(io.StringIO(text)))
head = next(i for i, r in enumerate(rows)
            if "Date" in r and "Topic" in r)
d_col = rows[head].index("Date")
t_col = rows[head].index("Topic")

def when(day):
    return f"{day:%A}, {day:%B} {day.day}"

schedule = []                       # (date, topic)
for r in rows[head + 1:]:
    if r[d_col].strip() and r[t_col].strip():
        day = datetime.strptime(r[d_col].strip(),
                                "%m/%d/%y")
        schedule.append((day, r[t_col].strip()))
docs = [f"{when(d)}: {t}" for d, t in schedule]
print(len(docs), "documents, e.g.", docs[-3])

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-retrieve.1.png?v=c0d666063c" class="img-responsive"/>
</div>

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
D = torch.tensor(embedder.encode(
    docs, normalize_embeddings=True))   # (docs, 384)

def retrieve(question, k=2):
    q = embedder.encode([question],
                        normalize_embeddings=True)
    scores = D @ torch.tensor(q[0])     # cosine
    return [docs[i] for i in scores.topk(k).indices]

print(retrieve("When is the second exam?"))

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-rag-prompt.png?v=fd3708ddba" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-rag.1.png?v=314b7da652" class="img-responsive"/>
</div>

In [ ]:
def rag_prompt(question, context):
    lines = "\n".join(f"- {d}" for d in context)
    return ("Answer using the course schedule.\n\n"
            f"Schedule:\n{lines}\n\n"
            f"Question: {question}")

def rag(question, k=2):
    context = retrieve(question, k)
    return ask(rag_prompt(question, context))

for q in ["When is Exam 02?",
          "Is there class on November 25?",
          "When is the Navy game?"]:
    print(q)
    print("  no RAG:", ask(q))
    print("  RAG:   ", rag(q))

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-facts-not-behavior.png?v=d0dbe4123a" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-cost.png?v=aad8c98bb7" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-lora.png?v=e73af75043" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-lora-count.png?v=5231bf7b3b" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-lora-where.png?v=61d7d87589" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-lora.1.png?v=be93d265ee" class="img-responsive"/>
</div>

In [ ]:
class LoRA:
    """W = W0 + (alpha / r) B @ A for one weight matrix.
    W0 is frozen. Only A and B are trained."""
    def __init__(self, layer, name, r=8, alpha=16):
        self.where = (layer, name)  # AdamW skips tuples
        self.W0 = getattr(layer, name).clone()
        d_out, d_in = self.W0.shape
        dev = self.W0.device
        self.A = torch.randn(r, d_in, device=dev)
        self.A /= d_in ** 0.5
        self.B = torch.zeros(d_out, r, device=dev)
        self.scale = alpha / r

    def merge(self):
        """Put W0 + scale B @ A into the layer."""
        layer, name = self.where
        W = self.W0 + self.scale * self.B @ self.A
        setattr(layer, name, W)

    def off(self):
        """Back to the frozen W0: the original model."""
        layer, name = self.where
        setattr(layer, name, self.W0)

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-lora-grad.1.png?v=646a24d389" class="img-responsive"/>
</div>

In [ ]:
def adapter_backward(a):
    """The block's backward already made dW for the
    whole matrix. Chain rule through W0 + s B @ A:"""
    layer, name = a.where
    dW = getattr(layer, "d" + name)
    a.dB = a.scale * dW @ a.A.T     # (d_out, r)
    a.dA = a.scale * a.B.T @ dW     # (r, d_in)

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-attach.1.png?v=88994efdf2" class="img-responsive"/>
</div>

In [ ]:
def n_weights(model):
    """The embedding (tied to the LM head) and the
    7 matrices in each block."""
    total = model.token_embedding.W.numel()
    for block in model.transformer_blocks:
        sa, ffn = block.SA, block.FFN
        for W in [sa.Wq, sa.Wk, sa.Wv, sa.Wo,
                  ffn.gate.W, ffn.up.W, ffn.down.W]:
            total += W.numel()
    return total

TOTAL = n_weights(model)
adapters = []
for block in model.transformer_blocks:
    for name in ["Wq", "Wv"]:
        adapters.append(LoRA(block.SA, name, r=8))
n = sum(a.A.numel() + a.B.numel() for a in adapters)
print(f"model: {TOTAL / 1e6:.1f} M weights")
print(f"LoRA trains {n:,} = {n / TOTAL:.2%}")

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-data.1.png?v=41a0d6d6ca" class="img-responsive"/>
</div>

In [ ]:
# Our way: one sentence, the source in brackets,
# and no guessing when the schedule doesn't say.
NOT_THERE = "I don't see that in the course schedule."

def reply_when(day, topic):
    return f"{topic} is on {when(day)}. [schedule]"

def reply_what(day, topic):
    return f"{when(day)} is {topic}. [schedule]"

def example(question, context, reply):
    """Chat prompt + reply. Prompt targets are PAD:
    read by the model, never graded."""
    ids = tok.chat_prompt(rag_prompt(question, context))
    out = tok.encode(reply + "<|im_end|>")
    X = ids + out[:-1]
    Y = [PAD] * (len(ids) - 1) + out
    return X, Y

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-examples.1.png?v=a84526c7e1" class="img-responsive"/>
</div>

In [ ]:
import random
random.seed(10124)
topics = [t for _, t in schedule]
rows = [(d, t) for d, t in schedule
        if topics.count(t) == 1 and "," not in t]
random.shuffle(rows)
test_rows, train_rows = rows[:6], rows[6:]
OFF = ["When is spring break?", "Who teaches CSE 30124?",
       "When is Exam 03?", "When are office hours?",
       "When is Homework 12 due?", "Where is the class?"]
OFF_TEST = ["When is the Navy game?",
            "What room is the exam in?"]

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-examples-2.1.png?v=be6fcb4dd6" class="img-responsive"/>
</div>

In [ ]:
def build(rows, off):
    data = []
    for day, topic in rows:
        doc = f"{when(day)}: {topic}"
        other = [d for d in docs if d != doc]
        for q, reply in [
                (f"When is {topic}?",
                 reply_when(day, topic)),
                (f"What is on {when(day)}?",
                 reply_what(day, topic))]:
            ctx = [doc, random.choice(other)]
            random.shuffle(ctx)
            data.append(example(q, ctx, reply))
        # the right row wasn't retrieved: don't guess
        ctx = random.sample(other, 2)
        data.append(example(f"When is {topic}?", ctx,
                            NOT_THERE))
    for q in off:
        for _ in range(4):
            ctx = random.sample(docs, 2)
            data.append(example(q, ctx, NOT_THERE))
    return data

train = build(train_rows, OFF)
val = build(test_rows, OFF_TEST)
print(len(train), "train /", len(val), "val examples")

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-batch.1.png?v=d5e06214b3" class="img-responsive"/>
</div>

In [ ]:
def collate(batch):
    """Right-pad to the longest. PAD isn't graded."""
    T = max(len(x) for x, _ in batch)
    X = torch.full((len(batch), T), PAD, dtype=torch.long)
    Y = torch.full((len(batch), T), PAD, dtype=torch.long)
    for i, (x, y) in enumerate(batch):
        X[i, :len(x)] = torch.tensor(x)
        Y[i, :len(y)] = torch.tensor(y)
    return X.to(model.device), Y.to(model.device)

def one_hot(Y):
    V = model.config["vocab_size"]
    out = torch.zeros(*Y.shape, V, device=Y.device)
    return out.scatter_(2, Y.unsqueeze(-1), 1.0)

def val_loss(model, examples, B=8):
    losses = []
    for i in range(0, len(examples), B):
        X, Y = collate(examples[i:i + B])
        Y_hat = model.forward(X)
        L = model.cross_entropy(Y_hat, one_hot(Y))
        losses.append(L.item())
    return sum(losses) / len(losses)

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-helpers.1.png?v=92008fc88b" class="img-responsive"/>
</div>

In [ ]:
def lora_backward(model, Y_hat, Y, adapters):
    """Backward through all 28 blocks (every block has
    adapters), skipping dW of the frozen head."""
    mask = (Y.argmax(-1) != PAD).unsqueeze(-1).float()
    grad = (Y_hat - Y) * mask / mask.sum()
    grad = grad @ model.lm_head.W
    grad = model.final_ln.backward(grad)
    for block in reversed(model.transformer_blocks):
        grad = block.backward(grad)
    for a in adapters:
        adapter_backward(a)

def free_grads(model):
    """Drop the dW the frozen layers made (~1.8 GB)."""
    params = list(optimizers.find_parameters(model))
    for layer, name in params:
        delattr(layer, "d" + name)
    torch.cuda.empty_cache()

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-train.1.png?v=74eb0a8cac" class="img-responsive"/>
</div>

In [ ]:
print(f"before: val loss {val_loss(model, val):.3f}")
opt = optimizers.AdamW(adapters, lr=5e-4,
                       weight_decay=0.0)
for step in range(1, 121):
    X, Y = collate(random.sample(train, 8))
    Y = one_hot(Y)
    Y_hat = model.forward(X)
    loss = model.cross_entropy(Y_hat, Y)
    lora_backward(model, Y_hat, Y, adapters)
    opt.step()                  # updates A and B only
    for a in adapters:
        a.merge()
    if step % 20 == 0:
        print(f"step {step}: loss {loss.item():.3f}")
del X, Y, Y_hat
free_grads(model)
print(f"after: val loss {val_loss(model, val):.3f}")

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-lora-merge.png?v=404d1ccd2a" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-swap.1.png?v=b6cd874e1f" class="img-responsive"/>
</div>

In [ ]:
# Questions it never trained on, with retrieval
tests = [f"When is {t}?" for _, t in test_rows[:3]]
tests += OFF_TEST
for a in adapters:
    a.off()
before = [rag(q) for q in tests]
for a in adapters:
    a.merge()
after = [rag(q) for q in tests]
for q, b, a in zip(tests, before, after):
    print(q)
    print("  adapter off:", b)
    print("  adapter on: ", a)

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-save.1.png?v=5fefb50549" class="img-responsive"/>
</div>

In [ ]:
# The adapter is all we need to keep
torch.save([(a.A.cpu(), a.B.cpu()) for a in adapters],
           "/content/assistant_lora.pt")
mb = os.path.getsize("/content/assistant_lora.pt") / 1e6
print(f"adapter: {mb:.1f} MB")
print(f"model:   {4 * TOTAL / 1e9:.1f} GB")

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-too-big.png?v=812f8e1827" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-distill.png?v=5fe52dccc7" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-hard-soft.png?v=5aca237f05" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-teacher.1.png?v=169fc9ec02" class="img-responsive"/>
</div>

In [ ]:
teacher = model               # our assistant, adapter on
q = "When is Review 01?"      # two rows say Review 01
print(retrieve(q))
ids = tok.chat_prompt(rag_prompt(q, retrieve(q)))
ids += tok.encode("Review 01 is on")
x = torch.tensor([ids], device=model.device)
probs = teacher.forward(x, eval=True)[0, -1]
top = probs.topk(5)
for p, i in zip(top.values, top.indices):
    word = tok.decode([i.item()])
    print(f"{word!r:>14}  {p.item():.3f}")

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-soft.png?v=963f402f78" class="img-responsive"/>
</div>

---

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-student.1.png?v=0629af44b5" class="img-responsive"/>
</div>

In [ ]:
def make_student():
    """A tiny irishQwen with Qwen's vocabulary."""
    torch.manual_seed(0)      # both students start equal
    return IrishQwen(ctx_len=512, d_model=256, n_layers=4,
                     n_heads=4, n_kv_heads=2, head_dim=64,
                     ffn_dim=768)

s = make_student()
head = s.lm_head.W.numel()   # untied in a new irishQwen
print(f"student: {(n_weights(s) + head) / 1e6:.1f} M")
emb = s.token_embedding.W.numel() + head
print(f"  embedding + LM head: {emb / 1e6:.1f} M")
print(f"teacher: {TOTAL / 1e6:.1f} M")
del s

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-distill.1.png?v=57f7718f9f" class="img-responsive"/>
</div>

In [ ]:
def train_student(student, soft, steps=150):
    opt = optimizers.AdamW(student, lr=1e-3,
                           weight_decay=0.0)
    gen = random.Random(10124)   # same batches for both
    for step in range(1, steps + 1):
        X, Y = collate(gen.sample(train, 8))
        graded = Y != PAD
        Y = one_hot(Y)           # the one right token
        if soft:
            P = teacher.forward(X)
            Y[graded] = P[graded]   # its probabilities
            del P
        Y_hat = student.forward(X)
        student.backward(Y_hat, Y)  # Y_hat - Y either way
        opt.step()
        if step % 50 == 0:
            L = val_loss(student, val)
            print(f"  step {step}: val loss {L:.3f}")
    return student

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-pd-distill-run.1.png?v=de0089f62b" class="img-responsive"/>
</div>

In [ ]:
print("hard labels")
hard = train_student(make_student(), soft=False)
L_hard = val_loss(hard, val)
del hard                      # room on the GPU
print("teacher's probabilities")
soft = train_student(make_student(), soft=True)

print(f"\nteacher        {val_loss(teacher, val):.3f}")
print(f"hard labels    {L_hard:.3f}")
print(f"distilled      {val_loss(soft, val):.3f}")

<div class="thumbnail">
    <img src="https://ai.williamtheisen.com/ballm/rag-lora/nb/lec-which.png?v=51d78d3890" class="img-responsive"/>
</div>